# Exp 026a: Torch vs TorchScript HGNet Benchmark

Benchmark CPU inference speed for our local `exp_011` HGNetV2 checkpoint using eager Torch and `torch.jit.trace` / `optimize_for_inference`.

## Plan

1. Load one local `exp_011` fold checkpoint.
2. Build a fixed Mel cache from a configurable set of soundscape files.
3. Benchmark eager Torch and TorchScript on the **same** cached Mel batches.
4. Save timing summaries and output-difference diagnostics for later deployment decisions.

In [1]:
EXPERIMENT_ID = 'exp_026a'
EXPERIMENT_NAME = 'torchscript_hgnet_benchmark'

from __future__ import annotations

import gc
import json
import math
import os
import statistics
import time
from dataclasses import asdict, dataclass
from pathlib import Path
import typing as tp

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from tqdm.auto import tqdm

ROOT = Path('/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026')
DATA = ROOT / 'data' / 'birdclef-2026'
EXPERIMENTS = ROOT / 'experiments'
EXP011_DIR = EXPERIMENTS / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised'


@dataclass
class BenchConfig:
    experiment_id: str
    experiment_name: str
    fold: int = 0
    source: str = 'train_soundscapes'
    n_files: int = 64
    batch_files: int = 4
    benchmark_repeats: int = 3
    benchmark_threads: tuple[int, ...] = (1, 2, 4, 8)

    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)

    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    pretrained: bool = False
    drop_path_rate: float = 0.0

    @property
    def clip_samples(self) -> int:
        return int(self.sample_rate * self.segment_seconds)

    @property
    def output_dir(self) -> Path:
        out = EXPERIMENTS / 'outputs' / f'{self.experiment_id}_{self.experiment_name}' / f'fold_{self.fold:02d}'
        out.mkdir(parents=True, exist_ok=True)
        return out


CFG = BenchConfig(experiment_id=EXPERIMENT_ID, experiment_name=EXPERIMENT_NAME)
SAMPLE_SUB = pd.read_csv(DATA / 'sample_submission.csv')
PRIMARY_LABELS = SAMPLE_SUB.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)


def resolve_checkpoint(fold: int) -> Path:
    path = EXP011_DIR / f'fold_{fold:02d}' / 'best_model.pt'
    if not path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {path}')
    return path


def resolve_source_paths() -> list[Path]:
    if CFG.source == 'train_soundscapes':
        root = DATA / 'train_soundscapes'
    elif CFG.source == 'test_soundscapes':
        root = DATA / 'test_soundscapes'
    else:
        raise ValueError(f'Unknown source: {CFG.source}')
    paths = sorted(root.glob('*.ogg'))
    if CFG.n_files is not None:
        paths = paths[: CFG.n_files]
    if not paths:
        raise FileNotFoundError(f'No .ogg files found under {root}')
    return paths


def read_soundscape_full(path: Path) -> np.ndarray:
    with sf.SoundFile(path) as snd:
        wave = snd.read(dtype='float32', always_2d=True)
    wave = wave.mean(axis=1).astype(np.float32, copy=False)
    return np.nan_to_num(wave, nan=0.0, posinf=0.0, neginf=0.0)


def slice_soundscape(audio: np.ndarray) -> np.ndarray:
    total = CFG.clip_samples * 12
    if len(audio) < total:
        audio = np.pad(audio, (0, total - len(audio)))
    else:
        audio = audio[:total]
    return audio.reshape(12, CFG.clip_samples).astype(np.float32, copy=False)


def iter_wave_batches(paths: list[Path], batch_files: int):
    for start in range(0, len(paths), batch_files):
        batch_paths = paths[start:start + batch_files]
        parts = [slice_soundscape(read_soundscape_full(path)) for path in batch_paths]
        wave_batch = np.concatenate(parts, axis=0)
        yield batch_paths, wave_batch


class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            pad_mode='reflect',
            norm='slaney',
            mel_scale='htk',
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        if wave.ndim == 1:
            wave = wave.unsqueeze(0)
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        flat = mel.flatten(start_dim=1)
        min_val = flat.min(dim=1).values[:, None, None, None]
        max_val = flat.max(dim=1).values[:, None, None, None]
        mel = (mel - min_val) / (max_val - min_val + 1e-7)
        return mel


def build_model() -> nn.Module:
    return timm.create_model(
        CFG.model_name,
        pretrained=CFG.pretrained,
        in_chans=1,
        num_classes=N_CLASSES,
        drop_path_rate=CFG.drop_path_rate,
    )


def load_fold_model() -> nn.Module:
    ckpt_path = resolve_checkpoint(CFG.fold)
    payload = torch.load(ckpt_path, map_location='cpu')
    state = payload.get('model_state_dict') or payload.get('state_dict') or payload
    model = build_model()
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


def set_cpu_threads(num_threads: int) -> int:
    max_threads = os.cpu_count() or num_threads
    threads = max(1, min(int(num_threads), max_threads))
    torch.set_num_threads(threads)
    return threads


SOURCE_PATHS = resolve_source_paths()
OUTPUT_DIR = CFG.output_dir
print({
    'output_dir': str(OUTPUT_DIR),
    'source': CFG.source,
    'n_files': len(SOURCE_PATHS),
    'n_waves': len(SOURCE_PATHS) * 12,
    'checkpoint': str(resolve_checkpoint(CFG.fold)),
})


{'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_026a_torchscript_hgnet_benchmark/fold_00', 'source': 'train_soundscapes', 'n_files': 64, 'n_waves': 768, 'checkpoint': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_011_hgnetv2_soundscape_supervised/fold_00/best_model.pt'}


In [2]:
MEL_TRANSFORM = LogMelSpectrogramTransform().cpu().eval()


def build_mel_batches(paths: list[Path]) -> list[np.ndarray]:
    mel_batches: list[np.ndarray] = []
    build_rows = []
    t0 = time.perf_counter()
    for batch_paths, wave_batch in tqdm(list(iter_wave_batches(paths, CFG.batch_files)), desc='build_mel_batches'):
        wave_t = torch.from_numpy(wave_batch)
        with torch.no_grad():
            mel = MEL_TRANSFORM(wave_t).cpu().numpy().astype(np.float32, copy=False)
        mel_batches.append(mel)
        build_rows.append({
            'batch_files': len(batch_paths),
            'batch_waves': int(wave_batch.shape[0]),
            'batch_mel_shape': list(mel.shape),
        })
    elapsed = time.perf_counter() - t0
    summary = {
        'n_batches': len(mel_batches),
        'n_files': len(paths),
        'n_waves': int(sum(batch.shape[0] for batch in mel_batches)),
        'build_seconds': elapsed,
    }
    (OUTPUT_DIR / 'mel_cache_summary.json').write_text(json.dumps(summary, indent=2))
    pd.DataFrame(build_rows).to_csv(OUTPUT_DIR / 'mel_batch_rows.csv', index=False)
    return mel_batches


MEL_BATCHES = build_mel_batches(SOURCE_PATHS)
print(json.dumps(json.loads((OUTPUT_DIR / 'mel_cache_summary.json').read_text()), indent=2))


build_mel_batches:   0%|          | 0/16 [00:00<?, ?it/s]

{
  "n_batches": 16,
  "n_files": 64,
  "n_waves": 768,
  "build_seconds": 4.236425000002782
}


In [3]:
def summarize_runs(name: str, thread_rows: list[dict[str, tp.Any]]) -> pd.DataFrame:
    df = pd.DataFrame(thread_rows)
    if not len(df):
        return df
    df = df.sort_values(['backend', 'threads']).reset_index(drop=True)
    return df


def check_output_diff(ref: np.ndarray, other: np.ndarray) -> dict[str, float]:
    diff = np.abs(ref - other)
    return {
        'max_abs_diff': float(diff.max()),
        'mean_abs_diff': float(diff.mean()),
    }


BASE_MODEL = load_fold_model().cpu().eval()
EXAMPLE_INPUT = torch.randn(8, 1, *CFG.image_size)

with torch.no_grad():
    eager_ref = BASE_MODEL(torch.from_numpy(MEL_BATCHES[0][:8]))

TRACED_MODEL = torch.jit.trace(BASE_MODEL, EXAMPLE_INPUT, strict=False)
TRACED_MODEL = torch.jit.optimize_for_inference(TRACED_MODEL.eval())
with torch.no_grad():
    traced_ref = TRACED_MODEL(torch.from_numpy(MEL_BATCHES[0][:8]))

CONSISTENCY = {
    'eager_vs_traced': check_output_diff(eager_ref.detach().cpu().numpy(), traced_ref.detach().cpu().numpy())
}
(OUTPUT_DIR / 'consistency_checks.json').write_text(json.dumps(CONSISTENCY, indent=2))
print(json.dumps(CONSISTENCY, indent=2))


{
  "eager_vs_traced": {
    "max_abs_diff": 2.09808349609375e-05,
    "mean_abs_diff": 4.1570419853087515e-06
  }
}


In [4]:
def run_torch_backend(model: nn.Module, mel_batches: list[np.ndarray]) -> tuple[float, int]:
    t0 = time.perf_counter()
    total_rows = 0
    with torch.no_grad():
        for mel in mel_batches:
            logits = model(torch.from_numpy(mel))
            total_rows += int(logits.shape[0])
    return time.perf_counter() - t0, total_rows


def benchmark_backend(backend_name: str, model_factory) -> list[dict[str, tp.Any]]:
    rows = []
    for thread_count in CFG.benchmark_threads:
        threads = set_cpu_threads(thread_count)
        model = model_factory()
        warmup_elapsed, warmup_rows = run_torch_backend(model, MEL_BATCHES[:1])
        run_seconds = []
        total_rows = 0
        for repeat_idx in range(CFG.benchmark_repeats):
            gc.collect()
            elapsed, total_rows = run_torch_backend(model, MEL_BATCHES)
            run_seconds.append(float(elapsed))
        rows.append({
            'backend': backend_name,
            'threads': threads,
            'warmup_seconds': float(warmup_elapsed),
            'rows': int(total_rows),
            'mean_seconds': float(statistics.mean(run_seconds)),
            'std_seconds': float(statistics.pstdev(run_seconds)) if len(run_seconds) > 1 else 0.0,
            'min_seconds': float(min(run_seconds)),
            'max_seconds': float(max(run_seconds)),
            'rows_per_second': float(total_rows / statistics.mean(run_seconds)),
            'run_seconds': json.dumps([round(x, 6) for x in run_seconds]),
        })
    return rows


eager_rows = benchmark_backend('torch_eager', lambda: BASE_MODEL)
traced_rows = benchmark_backend('torchscript_trace', lambda: TRACED_MODEL)
results = summarize_runs('torchscript', eager_rows + traced_rows)
results.to_csv(OUTPUT_DIR / 'variant_results.csv', index=False)

best_eager = float(results.loc[results['backend'].eq('torch_eager'), 'mean_seconds'].min())
best_traced = float(results.loc[results['backend'].eq('torchscript_trace'), 'mean_seconds'].min())
report = {
    'experiment_id': EXPERIMENT_ID,
    'experiment_name': EXPERIMENT_NAME,
    'fold': CFG.fold,
    'source': CFG.source,
    'n_files': len(SOURCE_PATHS),
    'n_waves': len(SOURCE_PATHS) * 12,
    'best_torch_eager_seconds': best_eager,
    'best_torchscript_seconds': best_traced,
    'best_torchscript_speedup': float(best_eager / best_traced) if best_traced > 0 else None,
    'consistency': CONSISTENCY,
    'output_dir': str(OUTPUT_DIR),
}
(OUTPUT_DIR / 'report_snapshot.json').write_text(json.dumps(report, indent=2))
print(results)
print(json.dumps(report, indent=2))


             backend  threads  warmup_seconds  rows  mean_seconds  \
0        torch_eager        1        3.253918   768     48.371688   
1        torch_eager        2        2.935383   768     47.347844   
2        torch_eager        4        2.904660   768     46.311929   
3        torch_eager        8        2.901818   768     45.845362   
4  torchscript_trace        1        2.962859   768     46.749540   
5  torchscript_trace        2        2.877083   768     45.881557   
6  torchscript_trace        4        2.839443   768     45.432292   
7  torchscript_trace        8        2.808451   768     44.941313   

   std_seconds  min_seconds  max_seconds  rows_per_second  \
0     0.173516    48.126503    48.502944        15.877056   
1     0.144470    47.229188    47.551214        16.220379   
2     0.033133    46.267577    46.347196        16.583200   
3     0.077542    45.761858    45.948672        16.751967   
4     0.097143    46.662841    46.885181        16.427969   
5     0.0225